## Narrow band FM demodulation and audio playback from a complex baseband IQ signal

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from wavinfo import WavInfoReader
import sounddevice as sd
import scipy.signal as signal
import scipy.fft as spfft
from numpy.polynomial.polynomial import polyval

In [2]:
# Interactive plotting.  Comment out this next line if inline plots are desired.
%matplotlib qt

In [3]:
# Function to plot I and Q time series from complex baseband IQ data.
#
# Inputs:
#   data - complex IQ signal to plot, or tuple with (I_data, Q_data)
#   nsamp - Number of samples to plot (starting at beginning)
#   ttext - title of plot
#   xlim - x axis plot limits: (min, max).  If None, autoscale
#   ylim - y axis plot limits: (min, max).  If None, autoscale
#
# Returns: 
#   Plot of time domain representation of complex baseband IQ signal.
#
# Affects: None
#
# Exceptions: None
#
def plot_complex_timeseries(data, nsamp, ttext, xlim=None, ylim=None):

    fig, (ax1, ax2) = plt.subplots(2, 1, sharey=True) # Share the y-axis

    # Plot I (ax1), first nsamp samples
    if len(data) == 2:
        # (I, Q) input
        ax1.plot(data[0][0:nsamp], color='blue', label='I')
    else:
        # WAV version with channel 1 = I, channel 2 = Q
        ax1.plot(data[0:nsamp,0], color='blue', label='I')
    ax1.set_title(ttext)
    ax1.set_xlabel('I value')
    ax1.set_ylabel('Amplitude')
    if xlim:
        ax1.set_xlim(xlim)
    if ylim:
        ax1.set_ylim(ylim)
    ax1.grid()
    ax1.legend()

    # Plot Q (ax2), first 4000 samples
    if len(data) == 2:
        # (I, Q) input
        ax2.plot(data[1][0:nsamp], color='red', label='Q')
    else:
        # WAV version with channel 1 = I, channel 2 = Q
        ax2.plot(data[0:nsamp,1], color='red', label='Q')
    ax2.set_xlabel('Q value')
    ax2.set_ylabel('Amplitude')
    if xlim:
        ax2.set_xlim(xlim)
    if ylim:
        ax2.set_ylim(ylim)
    ax2.grid()
    ax2.legend()

    # Adjust layout to prevent overlapping titles/labels
    plt.tight_layout()
    plt.show()                       

In [4]:
# Function to plot the frequency domain spectrum of a complex signal.
#
# Inputs: 
#  y - complex time domain signal to be plotted
#  fs - sampling rate, Hz
#  ttext - title of plot
#  xlim - x axis plot limits: (min, max)  None = autoscale
#  ylim - y axis plot limits: (min, max)  None = autoscale
#
# Returns:
#  Plot of frequency domain representation of signal.
#
# Affects: None
#
# Exceptions: None
#
def spec_plot_complex(y, fs, ttext, xlim=None, ylim=None):
    delta_t = 1.0 / fs
    no_samps = len(y)
    yf = spfft.fft(y)
    xf = spfft.fftfreq(no_samps, delta_t)
    xf_shift = spfft.fftshift(xf)
    yf_shift = spfft.fftshift(yf)
    plt.figure()
    plt.plot(xf_shift, 10 * np.log10(np.abs(yf_shift)))
    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)
    plt.xlabel('Frequency, Hz')
    plt.ylabel('Spectral amplitude')
    plt.title(ttext)
    plt.grid()
    plt.show()

In [5]:
# Function to plot the frequency domain spectrum of a real signal.
#
# Inputs: 
#  y - complex time domain signal to be plotted
#  fs - sampling rate, Hz
#  ttext - title of plot
#  xlim - x axis plot limits: (min, max)  None=autoscale
#  ylim - y axis plot limits: (min, max)  None=autoscale
#
# Returns:
#  Plot of frequency domain representation of signal.
#
# Affects: None
#
# Exceptions: None
#
def spec_plot_real(y, fs, ttext, xlim=None, ylim=None):
    delta_t = 1.0 / fs
    no_samps = len(y)
    yf = spfft.rfft(y)
    xf = spfft.rfftfreq(no_samps, delta_t)
    plt.figure()
    plt.plot(xf, 10 * np.log10(np.abs(yf)))
    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)
    plt.xlabel('Frequency, Hz')
    plt.ylabel('Spectral amplitude')
    plt.title(ttext)
    plt.grid()
    plt.show()

In [6]:
# Function to design a lowpass finite impulse response (FIR) filter design 
#  with linear phase response, using the window method with "firwin".
#  The filter is intended to be used with baseband complex signals.
#
# Inputs: 
#  bandwidth - Bandwidth of lowpass filter in Hz,
#              defined at the half-amplitude points (attenuation -6 dB).
#  fs - Sampling rate, Hz
#
# Returns:
#  h_lp - FIR filter coefficients as array with 251 tap entries.
#
# Affects: None
#
# Exceptions: None
#
def filt_def(bandwidth, fs):
    
    # This is a filter to be used on a complex baseband signal, so lowpass cutoff
    #  is set to half the desired bandwidth.
    f_lp_cutoff = bandwidth / 2.0

    # fixed number of taps for filter order
    num_taps = 251 # Filter order (must be odd for linear phase, which helps in design)

    # design lowpass filter
    h_lp = signal.firwin(num_taps, f_lp_cutoff, fs=fs, pass_zero='lowpass')

    # Filter is complete    
    return h_lp

In [7]:
# Function to calculate filter frequency response of a FIR filter.
#  Note: this is for visualization purposes only - the time domain filter coefficients
#   do the work here for the actual filtering further down, through the "lfilter" operator.
#
# Inputs: 
#  b - numerator coefficients of filter
#  sample_rate   - Sampling rate of filter, Hz
#  
# Returns:
#  freqs - Frequencies at which filter response was computed, Hz
#      range: [-sample_rate/2, +sample_rate/2)
#  H - Filter frequency response, as complex numbers
#
# Affects: None
#
def comp_fil_response(b, sample_rate):
    
    # Directly compute filter response.
    #
    # For more background on why the frequencies are first computed as radians on (-pi, pi] range, 
    # consult a DSP textbook for an explanation of z-transforms - e.g.
    #    Oppenheim & Schafer, Discrete-Time Signal Processing (3rd ed.)
    #    Proakis & Manolakis, Digital Signal Processing: Principles, Algorithms, and Applications
    #    Lyons, Understanding Digital Signal Processing
    #
    
    # Reverse the coefficients due to library incompatibilities:
    #
    # At present, "polyval" requires ordering of numerator coefficients 
    # from the lowest degree (i.e. the constant term) up to the highest degree.
    # However, the filtering operator lfilter() expect the numerator and denominator 
    # in highest to lowest degree form.
    #
    # This is true for numpy version 1.22.2, the version used at the time of writing,
    #   and for scipy version 1.8.0.  It may change in the future; beware.
    #  
    b = b[::-1]

    # (no numerator coefficients because for a FIR filter, the denominator is a constant = 1)
    
    # compute at 2048 frequencies for good fidelity
    w = np.linspace(-np.pi, np.pi, 2048, endpoint=False)
    
    # main loop
    z = np.exp(1j * w)
    num = np.zeros(len(z), dtype=np.complex64)
    for ii in range(len(z)):
        num[ii] = polyval(z[ii], b)
    H = num   # no denominator term = 1

    # convert frequencies to Hz rather than radians
    freqs = w * sample_rate / (2 * np.pi) # convert from radians to freq

    return freqs, H

In [8]:
# Function to plot the frequency domain response of a filter.
#
# Inputs: 
#  w - array of frequencies at which filter response h was computed.
#  H - array of frequency response, as complex numbers.
#  ttext - title of plot
#  xlim - x axis plot limits: (min, max)  None=autoscale
#  ylim - y axis plot limits: (min, max)  None=autoscale
#
# Returns:
#  Plot of abs(filter response) vs. real frequency.
#
# Affects: None
#
# Exceptions: None
#
def plot_filt(w, H, ttext, xlim=None, ylim=None):
    plt.figure()
    plt.plot(np.real(w), np.abs(H))
    if xlim:
        plt.xlim(xlim)
    if ylim:
        plt.ylim(ylim)
    plt.xlabel("Frequency, Hz")
    plt.ylabel("Magnitude")
    plt.title(ttext)
    plt.grid()
    plt.show()

In [9]:
# path and file name for the iq wav file
#   Acknowledgments: Paul Maine, KI5MIV, “Paul the SDR Guy”, 
#   for the audio file used for this FM demodulation.
file_path = "HamSCI_nbfm_modulate_iq.wav"

In [10]:
# set the sampling frequency for audio output
#  typical values might be 48000 Hz and 44100 Hz.
fs_audio = 48000

In [11]:
# read metadata information on input complex baseband IQ data
info = WavInfoReader(file_path)
print('File size: ' + str(info.data.byte_count)) # Accurate even for >4GB files
print('Metadata: ' + str(info.fmt))

File size: 18630864
Metadata: WavAudioFormat(audio_format=1, channel_count=2, sample_rate=250000, byte_rate=1000000, block_align=4, bits_per_sample=16)


In [12]:
# Assumes the recording is formatted as two channels (stereo) where channel 1 = I data
#  and channel 2 = Q data.
# See "scipy.wavfile" function documentation for information on manipulating WAV files in python.
#  Note that fs_rf = sampling rate of RF samples in baseband
fs_rf, data = wavfile.read(file_path)

In [13]:
# scale data to range between -1 and +1 (and implicitly conversation to floating point)
data = data / 32768.0 

In [14]:
# plot I and Q time series, first 4000 samples, from input wav file (complex baseband IQ data)
# autoscale
plot_complex_timeseries(data, 4000, 'Input complex baseband IQ signal')                

In [15]:
# form complex baseband IQ signal
signal_iq = data[:,0] + 1j * data[:,1]

In [16]:
# plot frequency response of input signal
spec_plot_complex(signal_iq, fs_rf, 'Freq response: Input complex baseband IQ signal')

In [17]:
# construct low pass FIR filter at 7000 Hz cutoff
h_lp = filt_def(7000, fs_rf)

In [18]:
# compute frequency response of the low pass FIR filter
freqs, H = comp_fil_response(h_lp, fs_rf)

In [19]:
# plot filter response
plot_filt(freqs, H, 'Lowpass FIR Filter Frequency Response (7000 Hz cutoff)')

In [20]:
# apply real filter to I and Q data separately
data_fil = np.empty_like(data)
a = [1.0]  # set so filter is a FIR and not IIR
data_fil[:,0] = signal.lfilter(h_lp, a, data[:,0])
data_fil[:,1] = signal.lfilter(h_lp, a, data[:,1])

In [21]:
# Since data is now low pass filtered,
# we can resample the signal at a lower rate than RF.
# first, compute integer ratio of sampling rates to use
gcd_fs = np.gcd(fs_rf, fs_audio)  # greatest common denominator for the two rates
up_conv = int(fs_audio / gcd_fs)
down_conv = int(fs_rf / gcd_fs)
print('Greatest common denominator: %i -> ratio = (%i / %i) (up_conversion, down_conversion)' % \
    (gcd_fs, up_conv, down_conv))

Greatest common denominator: 2000 -> ratio = (24 / 125) (up_conversion, down_conversion)


In [22]:
# plot I and Q time series, first 20000 samples, 
# from lowpassed version of input wav file (complex baseband IQ data)
# autoscale
plot_complex_timeseries(data_fil, 20000, 'LP Filtered complex baseband IQ signal')

In [23]:
# perform polynomial based resampling of low pass filtered complex baseband IQ data
# using up and down conversion integers computed earlier.  
# The result will be a new complex baseband IQ signal
# with a rate of fs_audio.
#  (see "resample_poly" for documentation on rational resampler)
f_poly_i = signal.resample_poly(data_fil[:,0], up_conv, down_conv)
f_poly_q = signal.resample_poly(data_fil[:,1], up_conv, down_conv)

In [24]:
# plot I and Q time series, first 4000 samples, 
# from resampled lowpassed version of input wav file (complex baseband IQ data)
# autoscale
plot_complex_timeseries((f_poly_i, f_poly_q), 4000, 'LP Filtered, decimated complex baseband signal')

In [25]:
# form complex baseband IQ signal from decimated lowpass filtered complex baseband IQ signal
data_iq = f_poly_i + 1j * f_poly_q

In [26]:
# Apply a hard limiter function
# to remove any AM modulation on the original FM signal from e.g. channel variations, etc.
mag = np.abs(data_iq)
# Set a floor value for any samples which are negative (which would distort the modulation)
mag = np.where(mag > 0, mag, 1e-6)
data_iq = data_iq / mag

In [27]:
# plot I and Q time series of the resampled data after hard limiter. 
# (demonstrates the FM nature of the original RF signal)
# autoscale
plot_complex_timeseries((np.real(data_iq),np.imag(data_iq)), 
                        4000, 'Filtered, decimated and hard limited complex baseband signal')

In [28]:
# plot frequency spectrum of the LP filtered / decimated / hard limited signal
spec_plot_complex(data_iq, 48000, 'Filtered, decimated and hard limited complex baseband signal')

In [29]:
# Now demodulate the narrow band FM (NBFM) signal,
# by differentiating the unwrapped phase of the signal in the time domain.
phase = np.unwrap(np.angle(data_iq))
demodulated = np.diff(phase)

In [30]:
# construct low pass FIR filter with 4000 Hz cutoff to apply to demodulated FM signal
h_lp_audio = filt_def(4000, fs_audio)

In [31]:
# calculate frequency response of LP filter
audio_freqs, audio_H = comp_fil_response(h_lp_audio, fs_audio)

In [32]:
# plot filter response
plot_filt(audio_freqs, audio_H, 'Lowpass FIR Filter Frequency Response Post-Demod (4000 Hz cutoff)')

In [33]:
# apply LP filter to demodulated audio
a = [1.0]  # set so filter is a FIR and not IIR
audio = signal.lfilter(h_lp_audio, a, demodulated)

In [34]:
# display time series of demodulated audio signal
plt.figure()
plt.plot(audio)
plt.xlabel('Sample number')
plt.ylabel('Amplitude')
plt.title('NBFM Demodulated Audio Time Series')
plt.grid()
plt.show()

In [35]:
# plot frequency spectrum of the LP filtered / decimated / hard limited signal
spec_plot_real(audio, fs_audio, 'Filtered, decimated and hard limited complex baseband signal') 

In [36]:
# play NBFM demodulated signal through computer audio
sd.play(audio, fs_audio)

In [37]:
# execute this cell to stop audio play
sd.stop()

In [38]:
# save NBFM demodulated signal as WAV file
wavfile.write("HamSCI_nbfm_demod.wav", fs_audio, audio)